# Statistical Arbitrage: Kalman-Filter Pairs Trading

This notebook walks end-to-end through the strategy: screen a universe for
cointegrated pairs, estimate a **dynamic** hedge ratio with a Kalman filter,
build a mean-reversion signal from the resulting spread, and backtest it
out-of-sample with realistic transaction costs.

The design choice that matters most is that everything is **causal and
out-of-sample**. Pairs are screened only on a formation window; the hedge ratio
and z-score at each day use only information up to that day; and performance is
reported only on the held-out trading window. The two mistakes that flatter
pairs backtests — look-ahead in the signal and free rebalancing of a dynamic
hedge — are both controlled for, and the correctness of the filter and the
PnL accounting is verified on synthetic data in `tests/`.

Run order: screen → pick a pair → filter → signal → backtest → read the
scorecard. The results tables below are intentionally left for you to fill by
running the cells on live data.

## 0. Setup

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Unzip the project and make its modules importable.
# Works in Colab and local Jupyter.
import os, sys, zipfile
from pathlib import Path

zip_path = Path('statarb-kalman-pairs.zip')
if not zip_path.exists():
    raise FileNotFoundError(
        'statarb-kalman-pairs.zip not found. Run the upload cell first, '
        'or place the zip in the same folder as this notebook.'
    )

extract_dir = Path('/content') if Path('/content').exists() else Path.cwd()
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

PROJECT_ROOT = extract_dir / 'statarb-kalman-pairs'
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError(f'Project folder not found after unzip: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Project extracted to:', PROJECT_ROOT)
print('Working directory:', Path.cwd())


In [ ]:
import os, sys
from pathlib import Path

# Find the project root whether this notebook is run from /content, notebooks/,
# or directly inside the extracted statarb-kalman-pairs folder.
candidates = [
    Path.cwd(),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else None,
    Path('/content/statarb-kalman-pairs'),
]
ROOT = next((p for p in candidates if p and (p / 'src').exists() and (p / 'config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find project root. Run the unzip cell first.')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import DEFAULTS, UNIVERSE
from src.data import load_prices, align_pair
from src.cointegration import screen_pairs, engle_granger, half_life
from src.kalman import KalmanHedge, rolling_ols_hedge
from src.signals import rolling_zscore, generate_positions, SignalConfig
from src.backtest import run_backtest, BacktestConfig
from src.metrics import summary

pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print('Imports loaded from:', ROOT)


## 1. Load prices and split formation vs trading

I pull adjusted closes and cache them to a parquet mirror, then cut the sample
into a **formation** half (used only for screening) and a **trading** half
(used only for evaluation). Nothing screened on the formation half has seen the
trading half.

In [ ]:
prices = load_prices(UNIVERSE, DEFAULTS["start"], DEFAULTS["end"])
split = int(len(prices) * DEFAULTS["formation_frac"])
formation, trading = prices.iloc[:split], prices.iloc[split:]

print(f"{prices.shape[1]} tickers, {len(prices)} rows")
print(f"formation: {formation.index[0].date()} -> {formation.index[-1].date()}")
print(f"trading:   {trading.index[0].date()} -> {trading.index[-1].date()}")
prices.tail()

## 2. Screen for cointegrated pairs (formation window only)

Correlation is a pre-filter, not the decision. The decision is the
**Engle-Granger** p-value (a residual ADF test on the static-OLS spread), with
the **Johansen** trace test as a multivariate cross-check and the OU
**half-life** to gauge whether reversion is fast enough to trade.

In [ ]:
pairs = screen_pairs(formation,
                     min_corr=DEFAULTS["min_corr"],
                     max_pvalue=DEFAULTS["max_pvalue"],
                     run_johansen=DEFAULTS["run_johansen"])
pairs

**Screening results** — fill by running the cell above:

| y | x | corr | EG p-value | hedge ratio | half-life | Johansen | passes |
|---|---|------|-----------|-------------|-----------|----------|--------|
|   |   |      |           |             |           |          |        |


## 3. Pick a pair and estimate the dynamic hedge ratio

I take the top-ranked pair (lowest Engle-Granger p-value) and run the Kalman
filter over the full sample so the hedge ratio is already warm when the trading
window begins. As a baseline I overlay a rolling-OLS hedge ratio — the naive way
to get a time-varying beta — which lags the Kalman estimate at turning points.

In [ ]:
top = pairs.iloc[0]
Y, X = top["y"], top["x"]
print(f"Trading pair: {Y} ~ {X}  (EG p={top['eg_pvalue']:.4f})")

logp = align_pair(prices, Y, X)
kf = KalmanHedge(delta=DEFAULTS["delta"], r=DEFAULTS["r"]).filter(logp[Y], logp[X])
ols_beta = rolling_ols_hedge(logp[Y], logp[X], window=DEFAULTS["zscore_window"])

fig, ax = plt.subplots(figsize=(11, 4))
kf.beta.plot(ax=ax, label="Kalman beta_t")
ols_beta.plot(ax=ax, label=f"rolling OLS ({DEFAULTS['zscore_window']}d)", alpha=0.7)
ax.axvline(prices.index[split], color="k", ls="--", lw=0.8, label="formation | trading")
ax.set_title(f"Dynamic hedge ratio: {Y} ~ {X}"); ax.legend(); plt.show()

## 4. Build the spread and the mean-reversion signal

The Kalman spread is the one-step forecast error `e_t`. I standardise it with a
**trailing** rolling window (causal) rather than the filter's own predicted
variance, because the trade is a bet on the spread *level* reverting. Entry,
exit and stop bands with hysteresis turn the z-score into positions.

In [ ]:
z = rolling_zscore(kf.spread, window=DEFAULTS["zscore_window"])
cfg_sig = SignalConfig(entry=DEFAULTS["entry"], exit=DEFAULTS["exit"], stop=DEFAULTS["stop"])
pos = generate_positions(z, cfg_sig)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
kf.spread.plot(ax=ax[0], title=f"Kalman spread e_t: {Y} ~ {X}")
z.plot(ax=ax[1], title="Rolling z-score with entry/exit bands")
for lvl, s in [(cfg_sig.entry,'-'),(-cfg_sig.entry,'-'),(cfg_sig.exit,':'),(-cfg_sig.exit,':')]:
    ax[1].axhline(lvl, color="grey", lw=0.7, ls=s)
plt.tight_layout(); plt.show()

## 5. Backtest out-of-sample with costs

Positions are lagged one day before meeting returns (execution lag), and costs
are charged on the full change in dollar weights each period — so daily
re-hedging of the dynamic beta is paid for, not assumed free. I report on the
trading window only.

In [ ]:
bt = run_backtest(logp, Y, X, pos, kf.beta,
                  BacktestConfig(cost_bps=DEFAULTS["cost_bps"],
                                 rebalance_hedge=DEFAULTS["rebalance_hedge"]))

trade_idx = prices.index[split:]
ret_oos = bt.returns.loc[trade_idx]
eq_oos = (1 + ret_oos).cumprod()
stats = summary(ret_oos, eq_oos, bt.positions.loc[trade_idx],
                bt.turnover.loc[trade_idx], DEFAULTS["periods_per_year"])
pd.Series(stats).to_frame("value")

**Out-of-sample scorecard** — fill by running the cell above:

| metric | value |
|--------|-------|
| ann_return |  |
| ann_vol |  |
| sharpe |  |
| sortino |  |
| max_drawdown |  |
| calmar |  |
| hit_rate |  |
| n_trades |  |
| avg_turnover |  |


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
eq_oos.plot(ax=ax)
ax.set_title(f"Out-of-sample equity (unit gross, net of {DEFAULTS['cost_bps']}bps): {Y} ~ {X}")
ax.set_ylabel("equity"); plt.show()

## 6. Cost sensitivity and the re-hedging question

A dynamic hedge ratio re-trades the hedge leg every day, so it is more exposed
to costs than a static hedge. Here I sweep the cost assumption and compare daily
re-hedging against freezing beta at entry, to see how much of any edge survives
friction. If the strategy only works at zero cost, that is worth knowing.

In [ ]:
rows = []
for cost in [0.0, 0.5, 1.0, 2.0, 5.0]:
    for rehedge in [True, False]:
        b = run_backtest(logp, Y, X, pos, kf.beta,
                         BacktestConfig(cost_bps=cost, rebalance_hedge=rehedge))
        r = b.returns.loc[trade_idx]
        e = (1 + r).cumprod()
        s = summary(r, e, b.positions.loc[trade_idx], b.turnover.loc[trade_idx],
                    DEFAULTS["periods_per_year"])
        rows.append({"cost_bps": cost, "rehedge": rehedge,
                     "sharpe": s["sharpe"], "ann_return": s["ann_return"],
                     "max_dd": s["max_drawdown"], "avg_turnover": s["avg_turnover"]})
pd.DataFrame(rows)

## 7. Notes and honest caveats

- **Survivorship / universe.** The universe in `config.py` is a hand-picked set
  of names that tend to pair; a production screen would run over a full,
  point-in-time-correct universe (including delisted names) to avoid selection
  bias in which pairs "happened" to cointegrate.
- **In-sample screening.** Cointegration is estimated on the formation window;
  a pair can and often does decohere out-of-sample. Reporting on the held-out
  window is the point, not a formality.
- **Costs.** 1bp per leg-trade is optimistic for anything but the most liquid
  names; the sensitivity sweep above is the honest way to see the break-even.
- **No metrics are hard-coded** anywhere in this repo. Every number comes from
  running these cells on data you supply.